In [1]:
import pandas as pd
import numpy as np
import pickle

oobasic = pd.read_excel('DATAFILES/oonames.xlsx')
pickle.dump(oobasic, open('HOME_PICKLE_FILES/oonames.pkl','wb'))

In [2]:
instlist = pd.read_pickle('HOME_PICKLE_FILES/listinstitutions.pkl')
oobasic = pd.read_pickle('HOME_PICKLE_FILES/oonames.pkl')
instlista = instlist[['ooidentifier','incitesname']]

In [3]:
import glob, os
os.chdir("INSTITUTIONAL_FILES")
files = sorted(glob.glob('*.txt'))

In [4]:
idatalist = []
names = []

In [5]:
for file in files:
    data = pd.read_csv(file, sep="\t",encoding='latin-1',low_memory=False)
    name = file.split('.')[0]
    names.append(name)
    if len(data.iloc[1, 0])>3:
        colnames = list(data.columns)
        colnames.append("final")
        data.columns = colnames[1:]
    data = data[['UT','SO','C1']]
    ndata = data['C1'].map(str)
    ndata = ndata.apply(lambda x: pd.Series(str(x).split("]")))
    ndata = ndata.apply(lambda x: x.astype(str).str.upper())
    sandata = pd.DataFrame(ndata)
    data = data[['UT','SO']]
    data = data.reset_index(drop=True)
    sdata = data.copy()
    todata = sdata.copy()
    for k in range(1,min(15,len(ndata.columns))):
        df = pd.DataFrame(sandata[k].str.split(',').str[0].str.strip())
        df = df.rename(columns={k:'oonames'})
        df = df.merge(oobasic, on='oonames',how='left')
        df = df.merge(instlista, on='ooidentifier',how='left')
        df = df.rename(columns={'incitesname':k})
        sdata = pd.concat([sdata, pd.DataFrame(df[k])], axis=1)
    sdata.insert(2, 'ninst', min(14, len(ndata.columns)-1) - sdata.isna().sum(axis=1))
    for k in range(1,min(15,len(ndata.columns))):
        df = sdata[['UT','SO','ninst',k]]
        df = df.rename(columns={k:'incitesname'})
        todata = pd.concat([todata,df],ignore_index=True)
    todata = todata.dropna()
    todata['ninst'] = todata['ninst'].astype(int)
    todata.sort_values('UT',ascending=True, inplace=True)
    todata = todata.reset_index(drop=True)
    todata = todata.rename(columns={'SO':'eco_incites'})
    todata = todata.rename(columns={'incitesname':'eco_institution'})
    todata = todata.reset_index(drop=True)
    todata = todata[['eco_incites','eco_institution','ninst']]
    idatalist.append(todata)    

In [6]:
os.chdir("..")
pickle.dump(idatalist, open('HOME_PICKLE_FILES/idatalist.pkl','wb'))
pickle.dump(files, open('HOME_PICKLE_FILES/iecofiles.pkl','wb'))
pickle.dump(names, open('HOME_PICKLE_FILES/ieconames.pkl','wb'))

In [7]:
print('The end')

The end
